# Prediction Deep Dive: Target, Baseline, Split, Metric, and Leakage

**Topic 12 · 1 lecture**

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/main/notebooks/figures/honr_464_logo.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>HONR 46400 — Evidence-Driven Research</center>
# <center>Professor: Davi Moreira</center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/blob/main/notebooks/student/nb12_prediction_student.ipynb)

---

## 🧭 Inquiry & Claim Boundary

**Inquiry emphasis:** **prediction**, the compass position where a descriptive
inquiry reaches all the way to cases you have not seen yet. In plain terms,
prediction asks: for a new case whose outcome is still unknown, what is my best
guess, and is that guess better than the dumbest honest rule? A weather app
answers a question like this every morning. Prediction forecasts; it does not
explain. This notebook also does one more thing, openly and on purpose: it
writes the design-library entry your book stops short of. RDSS lays out its
designs as a library (observational or experimental, each either descriptive or
causal, ch. 15–18) and treats prediction only in passing. So here you author
the missing entry, **"Observational: predictive,"** in the book's own
declare–diagnose–redesign grammar.

| | |
|---|---|
| **A claim this topic PERMITS** | "On held-out cases, my model beats the baseline by [margin] on [metric]." |
| **A claim this topic does NOT permit** | Any performance number computed on data the model already saw, and any "the model shows X *matters*" explanation read off the weights. That second one is a descriptive-to-causal crossing with no license. |

**Where this sits in the course:** the **Pilot, Diagnosis & Redesign** phase.
This topic develops the prediction branch of milestone **M09 (Pilot
Analysis)**, the pilot some projects will run and others will rule out with
reasons, which you present as a 4-minute walkthrough at the Friday studio.

*Provenance: RDSS Part III design library (ch. 15–18: observational/experimental × descriptive/causal) + §9.1.3 (answer strategy = the whole procedure) + ch. 10 (diagnosands) + ch. 11 (redesign) | prediction as the library entry the book stops short of | "Observational: predictive" declared in MIDA form: target/baseline/split/metric + a leaky-vs-clean demo | extended (course-authored library entry)*

## Learning Objectives

By the end of this notebook, you will be able to:

1. State the **prediction contract** (target, baseline, split, metric) and
   respect its order.
2. Score an honest baseline on held-out data, then read whether a model beats
   it and by how much.
3. Choose a metric that fits the target and defend the choice.
4. Detect leakage and quantify the damage by retraining on a clean feature set.
5. Separate prediction from explanation: say what a strong predictor can and
   cannot license about *why*.
6. Decide whether your own project has a predictive angle, or argue that
   prediction is the wrong compass position (the M09 prediction memo).

---

In [ ]:
# Setup — run this cell first.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 3)
plt.rcParams["figure.figsize"] = (9, 5)

SEED = 464  # course number — keeps every simulation reproducible
rng = np.random.default_rng(SEED)

# Data loads: GitHub raw URL first (works in Colab), local repo path as fallback.
DATA_URL = ("https://raw.githubusercontent.com/davi-moreira/"
            "2026F_evidence_driven_research_purdue_HONR464/main/notebooks/data/")

def load_course_data(filename):
    """Load a course dataset from GitHub, falling back to the local repo copy."""
    try:
        return pd.read_csv(DATA_URL + filename)
    except Exception:
        from pathlib import Path
        local = Path("notebooks/data") / filename
        if not local.exists():
            local = Path("../data") / filename
        return pd.read_csv(local)

# This notebook also uses scikit-learn for the prediction workflow.
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("✓ Setup complete — seed", SEED)

## 1. Why This Matters: The Contract Before the Code

> *"Everyone brings me a model that scores ninety-something percent. I have one
> question before I fund anything: ninety-something compared to WHAT? In my
> domain, guessing 'no' every single time already scores 88. If your model
> cannot beat that, you have not built anything. You have dressed up the base
> rate in a lab coat."*
> — a **policy stakeholder** who has been burned by impressive-sounding accuracy

Prediction is the compass position most likely to fool you, because a number
that *sounds* good can be worthless. The protection is a four-part
**contract**, and the order is not negotiable:

1. **Target**: the exact thing you are predicting, a single column defined
   before anything else. Ours today: did this person vote in 2012?
2. **Baseline**: the dumbest honest rule you must beat. Usually that rule is
   "always guess the most common answer."
3. **Split**: divide your rows into a training set the model learns from and a
   **held-out** set it never touches. Think of the held-out rows as exam
   questions the model never saw while studying.
4. **Metric**: how you keep score, chosen to match what the target is like.
   Accuracy, the share of guesses that were right, is one metric; you will
   meet others below.

Skip a step and the failure is predictable. No baseline, and you cannot tell
skill from the base rate. No held-out split, and you are grading the model on
its own homework. Wrong metric, and you optimize the wrong thing beautifully.

> **A question that often comes up here:** *"Isn't a higher accuracy always
> better than a lower one?"* No. Accuracy only means something *relative to
> the baseline*. If 88% of cases are "no," a model that says "no" to
> everything scores 88% while knowing nothing. The honest question is never
> "how high?" It is "how much higher than free?"

Our worked target is real. The Los Angeles voter file records, for 1,000
voters, whether each one **voted in 2012** (`voted_2012`), along with their
party, age, and census tract. You will try to predict turnout, and hold
yourself to the contract the whole way.

---

> 💡 **Gemini Prompt:** "Before building any model, this code loads a voter file and prints the target's base rate, the class balance, and how many categories the text columns have: [paste the next cell]. From the base rate I am about to see, explain what accuracy the 'always guess the most common answer' baseline would get by construction, and why a text column like `party` cannot be fed to a logistic model as-is."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Compute the majority-class baseline accuracy yourself from the printed base rate, then check Gemini's number against it. That figure is what any model must beat.
> - [ ] Confirm from the printed column types which features are text or codes and would need encoding first; don't take the AI's word for the data's shape.
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# Step 0 of any prediction task: LOOK at the data before modeling it.
voters = load_course_data("la_voter_file.csv")
assert voters.shape == (1000, 4), "unexpected shape — flag this!"
print("✓ Loaded la_voter_file.csv —", voters.shape[0], "rows,", voters.shape[1], "columns")
print()

print("Column types (a model cannot read text — categoricals will need encoding):")
print(voters.dtypes.to_string())
print()

base_rate = voters["voted_2012"].mean()
print(f"TARGET = voted_2012")
print(f"  base rate (share who voted): {base_rate:.1%}")
print(f"  class balance: {voters['voted_2012'].value_counts().to_dict()}")
print(f"  'party' has {voters['party'].nunique()} categories; "
      f"'census_tract' has {voters['census_tract'].nunique()} distinct codes.")

Three things in that printout matter before any model runs. First, the
majority class is "voted" (603 of 1,000), so the rule "always say voted"
scores 60.3% *by construction*. That is your baseline. Any model must beat it
to have earned anything. Second, `party` is text and `census_tract` is a code
wearing a number's clothes, so the code below one-hot encodes `party` (turns
each category into its own 0/1 column) before a model reads it. Third, at
603 to 397 the classes are mildly imbalanced, which means accuracy is readable
here but still hides *which* class the model gets right. A confusion matrix
will expose that hiding shortly.

**Section bridge:** before executing the contract, you will declare this study
the way your book declares every design. Prediction needs no new framework.

---

## 2. Declaring the Study: The Library Entry the Book Stops Short Of

RDSS's design library (Part III) has four rooms: observational or
experimental, each descriptive or causal (ch. 15–18). Prediction lives in
none of them by name; the book passes it by. But it is not homeless. On the
compass it is a descriptive inquiry whose reach is cases not yet seen, so its
natural address is the room the book left empty: **"Observational:
predictive."** In this section you furnish that room in the book's own
four-part grammar, **MIDA**.

<center><img src="https://raw.githubusercontent.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/main/notebooks/figures/rdss_fig_2_1.png" width="70%"/></center>

*MIDA: every declared design pairs the theoretical side (M, I) with the
empirical side (D, A). Prediction fills all four boxes; that is the whole
point. (Figure from the replication materials of Blair, Coppock & Humphreys
(2023), Research Design in the Social Sciences, an MIT-licensed archive; the
book is free at book.declaredesign.org.)*

**M, the model of the world.** A voter-file world: real registrants, each with
a party, an age, and a census tract, moving through an election in which some
turn out and some do not, all of it recorded on the rolls. This is the world
your forecast has to survive in.

**I, the inquiry.** Not a population average and not a cause. The quantity is
defined for units you have not seen: *for a registrant whose turnout is still
unknown, will they vote in the next election?* That is a descriptive question
aimed at unrealized cases, the compass's third reach made precise.

---

**D, the data strategy.** Here is where prediction earns its rigor. A forecast
is only worth anything if it uses information actually available at the moment
the forecast is needed, before the outcome exists. That single clause defines
the field's most famous mistake. **Leakage** is a feature that secretly
carries the answer: its value is only settled at or after the outcome it is
supposed to predict. Example: forecasting hospital readmission from a code
that is only recorded at discharge. In the book's grammar, leakage is a
data-strategy violation. You will build one yourself in §4 and watch it
corrupt a score.

**A, the answer strategy.** The book's sharpest rule (§9.1.3): your answer
strategy is the *entire* procedure you run. The learner, plus the baseline
you measure it against, plus every choice you made along the way. If you
quietly tried three feature sets and reported the best, your answer strategy
is "try three and keep the winner," not "fit one model." Declaring A honestly
means owning the whole path, because the whole path is what a skeptic has to
be able to rerun.

**Diagnosis and redesign.** Your book judges a design by **diagnosands**:
statistics you compute by running the design many times and watching how it
behaves (ch. 10). Prediction's diagnosis is the held-out test set. The test
set is a stand-in world the model never trained on, so held-out accuracy is a
diagnosand. And when you pit the model against the baseline, you are doing
**redesign** (ch. 11): holding the inquiry fixed and comparing candidate
designs on the same diagnosand. The baseline is the "do nothing clever"
variant every real design must beat.

> **A question that often comes up here:** *"Isn't this just machine learning
> with the book's words bolted on?"* It is the reverse. The book's words keep
> machine learning honest. Declaring I forces you to say the target is about
> *unseen* units, not the ones in hand. Declaring D turns "leakage" from a
> vague worry into a rule about timing. Declaring A makes you confess the
> whole selection procedure. And diagnosing on held-out data is the only thing
> that stops a memorized score from masquerading as skill. Prediction did not
> need a fifth framework; MIDA declares it too.

**Section bridge:** the declaration is on paper. Now you execute it.

---

## 3. The Prediction Contract, Executed

**Guiding question:** *is my model's score any better than the dumbest honest
rule, and is it scored on cases the model never saw?*

Now you honor all four parts. You split first: 25% of the rows go into a
held-out test set the model never sees during training. The split uses a
fixed seed so it is reproducible, and it is *stratified*, meaning both halves
keep the same turnout balance. This split is the diagnosis step from §2 made
concrete. Then you fit the baseline (a `DummyClassifier` that always predicts
the majority class) and a real model (`LogisticRegression` on the encoded
features), and you score *both* on the held-out rows with the same metric.
Pitting model against baseline is the first redesign comparison: same inquiry,
same diagnosand, two candidate designs.

The order protects you. Because the split happened before any fitting, every
number below is **out-of-sample**: computed only on cases the model never
trained on, the way a real forecast would be judged. Out-of-sample numbers
are the only kind the policy stakeholder will accept.

> **A question that often comes up here:** *"Why hold out 25% of my data?
> Isn't throwing away a quarter of it wasteful?"* The held-out rows are not
> thrown away; they are the only honest test you have. Train on every row and
> no cases are left that the model did not memorize, so its score measures
> recall of its own homework rather than skill on the unseen. Spending a
> quarter of the data to earn a number a skeptic will believe is the opposite
> of waste. It is what buys the number its meaning.

---

### 🔮 Pause & Predict

The baseline always guesses the majority answer, "this person voted," for
every held-out voter. **Before running the next cell**, write your
prediction: what accuracy will that baseline get on the held-out rows? Give a
number, and one sentence on why. (Hint: you saw the whole-file base rate a
moment ago.)

### YOUR ANSWER HERE:

**My predicted baseline accuracy (a number):**

**Why:**

---

### 🛠️ Run the Study — baseline first, then try to beat it

Run the cell below. It splits the data, scores the honest baseline on
held-out rows, then scores a logistic model on the *same* held-out rows.
Watch the two numbers, and the gap between them.

> 💡 **Gemini Prompt:** "This code holds out 25% of the rows (stratified, fixed seed), scores a majority-class baseline on them, then scores a logistic model on the SAME held-out rows: [paste the next cell]. Explain what `stratify=y` does and why the split has to happen before any fitting. Then predict whether the model's margin over the baseline will be large or small, and why 2012 turnout is hard to predict from party, age, and tract alone."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Check the printed baseline and model accuracies and the margin between them. Is the gain as large as Gemini guessed, or the modest few points the data actually support?
> - [ ] Confirm the held-out base rate the cell prints matches the whole-file base rate (that is stratification working), rather than trusting the AI's account of the split.
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# The reveal — run AFTER committing your prediction above.
y = voters["voted_2012"]
X = pd.get_dummies(voters[["party", "age", "census_tract"]],
                   columns=["party"], drop_first=True)

# SPLIT: hold out 25%, fixed seed, stratified so turnout balance is preserved.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=464, stratify=y)
print(f"train rows: {len(y_train)}   held-out rows: {len(y_test)}   "
      f"held-out base rate: {y_test.mean():.1%}")

# BASELINE: always predict the majority class.
baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
base_acc = accuracy_score(y_test, baseline.predict(X_test))

# MODEL: logistic regression on the encoded features (standardized).
model = make_pipeline(StandardScaler(),
                      LogisticRegression(max_iter=1000)).fit(X_train, y_train)
model_acc = accuracy_score(y_test, model.predict(X_test))

print()
print(f"  baseline held-out accuracy: {base_acc:.3f}")
print(f"  model    held-out accuracy: {model_acc:.3f}")
print(f"  margin (model - baseline):  {model_acc - base_acc:+.3f}")

In [ ]:
# Self-check: the honest guarantees the contract is supposed to give us.
assert set(X_test.index).isdisjoint(set(X_train.index)), \
    "held-out rows leaked into training — the split is broken!"
assert model_acc >= base_acc, "model did not beat the baseline — report that honestly, don't hide it"
print("✓ Self-check passed: test rows were never trained on, "
      f"and the model's {model_acc:.1%} is scored entirely out-of-sample.")
print(f"✓ The model beats the baseline by {model_acc - base_acc:.1%} — "
      "small, and that smallness is the lesson.")

### 🔍 Reading the Evidence — write both headlines

The model scored about **0.632**; the baseline scored about **0.604**. That
is a real edge of roughly three percentage points, and it is *small*. This is
the honest-beats-easy moment: a modest, true gap is worth more than a
dramatic, misleading one. In the cell below, write the **honest headline**
first. Then write the **dishonest headline** you could technically get away
with, and strike it through, so you feel exactly what you are refusing to
publish.

### YOUR ANSWER HERE:

**Honest headline (names the baseline and the margin):**

**Dishonest headline, struck through so it never ships:** ~~...~~

**The margin, in one number:**

---

### ⚖️ Make a Design Choice — pick the metric that matches the job

Accuracy is not the only scorecard, and it is often the wrong one. The cell
below prints a **confusion matrix**, a small table of truths versus guesses:
rows are what really happened, columns are what the model predicted. It also
prints each class's **recall** (of all the true cases of that class, what
share did the model catch?) and **precision** (of the cases the model flagged
as that class, what share truly were?). Run it and look at *which* voters the
model gets right and wrong.

In [ ]:
# Where do the model's right and wrong answers fall?
cm = confusion_matrix(y_test, model.predict(X_test))
print("Confusion matrix (rows = truth, cols = prediction):")
print(pd.DataFrame(cm,
      index=["truth: did NOT vote", "truth: voted"],
      columns=["pred: did NOT vote", "pred: voted"]))
print()
print(classification_report(y_test, model.predict(X_test),
      target_names=["did not vote (0)", "voted (1)"]))

Notice the asymmetry: the model catches most actual voters but only a small
share of actual non-voters. It leans hard toward the popular answer. Now
suppose you run a get-out-the-vote campaign and want to **find the
non-voters** so you can reach them. Accuracy would reward the model for
nailing the voters you were *not* trying to find.

In the cell below, choose the metric you would optimize for the campaign:
(a) overall accuracy, (b) recall on the non-voter class, or (c) precision on
the non-voter class. Defend the choice in one paragraph tied to what a wrong
answer *costs* the campaign.

**Section bridge:** the contract is signed and the metric chosen. Now meet
the signature way the contract gets broken.

### YOUR ANSWER HERE:

**Metric I would optimize (a / b / c):**

**One-paragraph defense (what a false positive vs a false negative costs here):**

---

## 4. Leakage: When a Great Score Is a Warning Sign

**Guiding question:** *when a score looks too good, has the model learned the
world, or quietly found the answer key?*

> *"When a model suddenly scores far better than anyone expected, I don't get
> excited. I get suspicious. Nine times out of ten it found a shortcut: a
> feature that already contains the answer. The tenth time is a real
> breakthrough. My job is to rule out the nine before we celebrate the one."*
> — a **skeptical peer** reviewing your pilot

Leakage (§2) is the quiet killer of prediction projects: a feature that
carries information you would *not* actually have at prediction time. Very
often it is the outcome itself, lightly disguised. It produces spectacular
held-out scores that evaporate the moment the model meets the real world,
because in the real world that feature is not available yet, or is only
knowable *because* the outcome already happened.

The best way to learn the hunt is to build a leak yourself and then catch it.
In the next cell you will construct a feature called `post_election_contact`
**from the very outcome you are predicting**, add it to the model, and
retrain. Your job is to feel how good the score looks, then find the trick.

> **A question that often comes up here:** *"If leakage makes the score go
> UP, why is it bad? Isn't a higher score what we want?"* The score goes up
> on *this* data and collapses on *new* data, because the leaky feature will
> not exist, or will not yet be known, when you actually need a forecast. A
> model that scores 90% by reading the answer key has learned nothing it can
> use on a case whose answer is still unknown. High-and-fake is worse than
> modest-and-real.

---

### 🔮 Pause & Predict

Imagine a canvasser's log flag, `post_election_contact`, tallied *after* the
election from the same rolls that record turnout. It is tangled up with
whether the person voted. The honest model scored about 63% held-out.
**Before running**, commit a number: what will held-out accuracy be once this
leaky feature joins the model? Roughly the same, a little higher, or a lot
higher?

### YOUR ANSWER HERE:

**My predicted held-out accuracy with the leaky feature:**

**Why:**

---

> 💡 **Gemini Prompt:** "This code deliberately builds a feature called `post_election_contact` FROM the outcome (`voted_2012`, with 15% of values flipped as noise), adds it to the model, and retrains: [paste the next cell]. Explain why a feature constructed from the outcome will make held-out accuracy jump, and predict roughly how high the accuracy will climb compared with the honest model's roughly 63%."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Check the printed leaky accuracy against Gemini's prediction. Did a feature that encodes the answer inflate the score as expected?
> - [ ] Remember the tell for real projects: no feature arrives labeled "leak." Confirm you could catch this one by noticing its value depends on the outcome, not by trusting a label.
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# You are building a leak ON PURPOSE so you can practice catching one.
# `post_election_contact` is built FROM voted_2012 — it is the outcome with 15%
# of its values flipped as noise. In a real project no feature arrives labeled
# "leak"; the tell is that its value depends on the outcome.
leak_rng = np.random.default_rng(SEED)
flip = leak_rng.random(len(voters)) < 0.15
voters["post_election_contact"] = np.where(flip, 1 - voters["voted_2012"],
                                           voters["voted_2012"])

feat = pd.get_dummies(voters[["party", "age", "census_tract"]],
                      columns=["party"], drop_first=True)
X_clean = feat
X_leaky = feat.assign(post_election_contact=voters["post_election_contact"].values)

Xc_tr, Xc_te, y_tr, y_te = train_test_split(
    X_clean, y, test_size=0.25, random_state=464, stratify=y)
Xl_tr, Xl_te, _, _ = train_test_split(
    X_leaky, y, test_size=0.25, random_state=464, stratify=y)

leaky_model = make_pipeline(StandardScaler(),
                            LogisticRegression(max_iter=1000)).fit(Xl_tr, y_tr)
leaky_acc = accuracy_score(y_te, leaky_model.predict(Xl_te))
print(f"  model WITH the leaky feature — held-out accuracy: {leaky_acc:.3f}")
print("  (that jump should make you suspicious, not proud)")

### 🛠️ Run the Study — hunt the leak, then retrain clean

The score just leapt from about 0.63 to about 0.89. A skeptic runs three
checks on any feature behind a suspicious jump:

1. **Too good to be true?** A large, sudden gain over a sensible baseline is
   a red flag, not a trophy.
2. **Suspiciously correlated with the target?** A single feature that tracks
   the outcome almost perfectly usually *is* the outcome.
3. **Known too late?** If the feature's value is only settled at or after the
   moment the outcome happens, it cannot be an input to a real forecast.

Run the cell below to apply checks 2 and 3, then retrain without the leak and
read the honest delta.

> 💡 **Gemini Prompt:** "This code checks a suspicious feature two ways, its correlation with the outcome and the timing of when its value is settled, then retrains without it and reports the honest change in accuracy: [paste the next cell]. Explain why a near-perfect correlation with the target, PLUS a value that is only known after the outcome, together prove a feature cannot be used in a real forecast."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Read the printed correlation and the clean-vs-leaky accuracy gap. Does dropping the leak collapse the score back toward the honest number, as it should?
> - [ ] Apply the timing test to a feature in your OWN data: is its value settled before or after the moment you would need the prediction? The AI cannot see your data's timeline.
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# CHECK 2 — how tightly does the suspect feature track the outcome?
r = np.corrcoef(voters["post_election_contact"], voters["voted_2012"])[0, 1]
print(f"corr(post_election_contact, voted_2012) = {r:.3f}   "
      "→ this feature is almost a copy of the answer")
# CHECK 3 — timing: it is tallied AFTER the election, so no real forecast could use it.
print("timing: recorded AFTER turnout is known → not available at prediction time")
print()

# RETRAIN CLEAN — drop the leak, score honestly again.
clean_model = make_pipeline(StandardScaler(),
                            LogisticRegression(max_iter=1000)).fit(Xc_tr, y_tr)
clean_acc = accuracy_score(y_te, clean_model.predict(Xc_te))
print(f"  leaky held-out accuracy: {leaky_acc:.3f}")
print(f"  clean held-out accuracy: {clean_acc:.3f}")
print(f"  inflation the leak bought: {leaky_acc - clean_acc:+.3f}")

In [ ]:
# Self-check: the leak really did inflate the score, and both numbers are honest.
assert leaky_acc > clean_acc, "the leak should inflate the score"
assert set(Xl_te.index).isdisjoint(set(Xl_tr.index)), "held-out rows leaked into training!"
assert set(Xc_te.index).isdisjoint(set(Xc_tr.index)), "held-out rows leaked into training!"
print(f"✓ Self-check passed: the leak inflated held-out accuracy by "
      f"{leaky_acc - clean_acc:.1%} — pure illusion, since the feature encodes the answer.")
print("✓ Both scores were computed only on rows the model never trained on.")

### 🔍 Reading the Evidence — prediction is not explanation

If you inspected the leaky model's coefficients, `post_election_contact`
would tower over every real feature. It would look like "the most important
driver of voting." It is nothing of the kind. It is the answer, copied. This
is the **performance-understanding gap**: a feature's weight or "importance"
tells you what the model *leaned on*, not what *causes* the outcome. A leak
fakes importance; even an honest predictor's weights are not mechanisms.

In the cell below, answer two things. First: what did the leaky model's
top-ranked feature teach you about *why* people vote? Second: state the exact
claim boundary. Write one sentence the clean model's 63% *permits*, and one
sentence it does *not* permit no matter how the coefficients rank.

### YOUR ANSWER HERE:

**What the leaky model's top feature taught me about why people vote:**

**A claim the clean model permits:**

**A claim it does NOT permit (an explanation read off the weights):**

---

### 📝 Practice — rank features by leakage risk

You are predicting, at the moment a patient is **admitted** to a hospital,
whether they will be **readmitted within 30 days of discharge**. Rank these
four features from highest to lowest leakage risk, and give the one-word
reason (timing) for the top and bottom.

- **A.** Patient's age at admission.
- **B.** Number of prior admissions in the previous year.
- **C.** Discharge disposition code (e.g., "transferred to another
  facility"), recorded *at discharge*.
- **D.** Total medications administered during the stay, finalized *at
  discharge*.


### YOUR ANSWER HERE:

**Ranking (highest → lowest leakage risk):**

**Reason for the highest / lowest:**

---

> 💡 **Gemini Prompt:** "Here is the target I want to predict: [describe your outcome and when it is measured]. Here is my candidate feature list: [list features]. For each feature, tell me at what moment its value becomes known relative to the outcome, and flag any that would only be knowable at or after the outcome occurs."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] For every feature the AI clears, check the timing yourself against when your outcome is actually recorded. The AI does not know your data's timeline.
> - [ ] Treat a "no leakage detected" verdict as a hypothesis to test, never a guarantee. You catch leaks by re-deriving each feature's timing, not by trust.
> - [ ] Log this use in your AI ledger: tool, task, verification.

### 🎯 Project Transfer — your M09 prediction memo

Milestone **M09 (Pilot Analysis)** branches by compass position, and this
memo settles its prediction branch. You will walk your pilot through in about
4 minutes at the Friday studio. In the cell below, decide whether your own
project has a predictive angle.

If it **does**: name the target (the exact column you would predict), the
baseline you would have to beat, the metric that fits the target's shape, and
the single feature in your data most likely to leak, with the concrete check
you will run to rule it out before you trust any score.

If it does **not**: write one honest sentence on why prediction is the wrong
compass position for your question (description, generalization, or causal
reasoning is what you actually want), then name the analogous
too-good-to-be-true trap in your actual approach (a measure that quietly
contains its own outcome) and your check for it. Both verdicts are correct if
they are defended; this memo is graded on the reasoning, not the verdict.

### YOUR ANSWER HERE:

**Does my project have a predictive angle? (yes / no):**

**If yes (target / baseline / metric):**

**The feature (or measure) most likely to leak + my concrete check:**

**If no (one sentence on why prediction is the wrong compass position):**

---

### 🎟️ Claim Ticket

**Claim Ticket #1** — your project's prediction verdict, in one defended
sentence: *"My project [has a prediction angle: target ___, baseline ___,
metric ___, nearest leakage risk ___] / [does not; prediction is the wrong
compass position because ___]."*

### YOUR ANSWER HERE:

**My prediction verdict, one sentence:**

---

## 5. If You Want to Go Further *(optional)*

Nothing in this section is required. It is extra range for the curious: two
more metric scenarios, and the ethics edge of prediction.

### 📝 Practice — match the metric to the target *(optional)*

Three prediction targets below. For each, name a metric you would trust and
one you would distrust, in a sentence. The shape of the target decides the
scorecard.

- **A.** Flagging fraudulent transactions, where about **1 in 1,000** is
  fraud.
- **B.** Predicting the winner of a near **50/50** two-option A/B test
  outcome.
- **C.** Flagging patients at risk of a **dangerous complication**, where
  missing a true case can be fatal but a false alarm just triggers a second
  check.


### YOUR ANSWER HERE:

**A (rare event):**

**B (balanced):**

**C (costly misses):**

---

### One legitimate use, one illegitimate use *(optional)*

Here is the ethical edge of prediction. Suppose an algorithm scores who is
likely to **miss a probation appointment**. The same scores can be used two
ways. In the cell below, write one **legitimate** use and one
**illegitimate** use, and name what makes the difference.

*(For the curious: the case-study index at*
[callingbullshit.org/case_studies.html](https://callingbullshit.org/case_studies.html)
*collects real examples of prediction dressed up as understanding. Browse the
index; no single case required.)*

### YOUR ANSWER HERE:

**Legitimate use of the scores:**

**Illegitimate use of the same scores:**

**What makes the difference:**

---

## 6. Wrap-Up

In one lecture you ran prediction the honest way. You signed the **contract**
(target, baseline, split, metric, in that order) and watched a logistic model
beat the majority-class baseline by a real but modest three points. That
taught you that *honest beats easy*: a small true edge is worth more than a
big misleading one. Then you built a leak, felt the fake 89%, hunted it down
by correlation and timing, retrained clean, and reported the delta. And you
drew the hard line: a model's weights are not mechanisms. Prediction answers
*who*, never *why*.

Notice what you never needed: a fifth framework. Prediction fit the book's
own **MIDA** grammar from the start. An inquiry about unseen units, a data
strategy that forbids leakage, an answer strategy that owns the whole
procedure, and a held-out diagnosis. That is exactly why this course could
write the missing library entry, "Observational: predictive," in the book's
own grammar rather than bolt a separate unit onto the side of the compass.

> **"A prediction score means nothing until you name the baseline it beat and
> prove the model never saw the answer."**

Next, nb13 chases the harder word. Prediction was content to forecast *who*;
causal reasoning insists on *because*. To earn that word you will need
counterfactuals, randomization, and an identification argument you can
defend. Bring your M09 pilot plan; the causal branch is where many projects
land.

---

## 7. Sources & Provenance

**Provenance of this notebook:**
- *RDSS Part III design library (ch. 15–18: observational/experimental × descriptive/causal) | prediction as the entry the book stops short of | "Observational: predictive" authored in the book's declare–diagnose–redesign grammar | extended (course-authored library entry)*
- *RDSS §9.1.3 (answer strategy = the whole procedure) + ch. 10 (diagnosands) + ch. 11 (redesign) + ch. 2 fig. 2.1 (MIDA) | MIDA declaration of the turnout study; held-out test as diagnosand; baseline/feature comparison as redesign | conceptual crosswalk (scikit-learn mechanics) | extended*
- *fresh (scikit-learn) | target/baseline/split/metric protocol + leaky-vs-clean leakage demo (leakage named as a data-strategy violation) | built for the course | fresh*
- *la_voter_file.csv | rdss package data | 2012 turnout target predicted from voter-file fields | adapted (used here as a prediction target; same file as nb07's sampling frame)*

**Dataset attribution:** Dataset from the `rdss` package (Blair, Coppock &
Humphreys, MIT License), companion to *Research Design in the Social
Sciences* (2023).

**Readings:**
- Blair, G., Coppock, A., & Humphreys, M. (2023). *Research Design in the
  Social Sciences*, Part III (the design library, ch. 15–18) + §9.1.3 (answer
  strategy) + ch. 10–11 (diagnosis and redesign). Free online:
  [book.declaredesign.org](https://book.declaredesign.org/). The mechanics
  (train/test, metrics, leakage) are built fresh with scikit-learn; the
  *frame* is the book's.
- Optional enrichment: the case-study index at
  [callingbullshit.org/case_studies.html](https://callingbullshit.org/case_studies.html)
  (index only) for real examples of prediction mistaken for understanding.

---

<center>

Thank you!

</center>